### ***Load and preprocess data***

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load datasets
books = pd.read_csv('../data/BX-Books.csv', sep=';', on_bad_lines='skip', encoding='latin-1')

C:\Users\emon1\AppData\Local\Temp\ipykernel_20328\3948767560.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('../data/BX-Books.csv', sep=';', on_bad_lines='skip', encoding='latin-1')


In [4]:
books.head(2)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...


In [5]:
books.shape

(271360, 8)

In [6]:
users = pd.read_csv('../data/BX-Users.csv', sep=';', on_bad_lines='skip', encoding='latin-1')

In [8]:
users.head(2)

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0


In [7]:
users.shape

(278858, 3)

In [9]:
ratings = pd.read_csv('../data/BX-Book-Ratings.csv', sep=';', on_bad_lines='skip', encoding='latin-1')

In [10]:
ratings.head(2)

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5


In [11]:
ratings.shape

(1149780, 3)

### ***Rename books data***

In [12]:
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-L']]

In [13]:
books.rename(columns={
        'Book-Title': 'title',
        'Book-Author': 'author',
        'Year-Of-Publication': 'year',
        'Publisher': 'publisher',
        'Image-URL-L': 'img_url'
    }, inplace=True)

C:\Users\emon1\AppData\Local\Temp\ipykernel_20328\869973553.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  books.rename(columns={


### ***Rename users data***

In [14]:
users.rename(columns={
        'User-ID': 'user_id',
        'Location': 'location',
        'Age': 'age'
    }, inplace=True)

### ***Rename ratings data***

In [15]:
ratings.rename(columns={
        'User-ID': 'user_id',
        'Book-Rating': 'rating'
    }, inplace=True)

### ***Filter active users (users with more than 200 ratings)***

In [16]:
user_ratings_count = ratings['user_id'].value_counts()

In [17]:
user_ratings_count

user_id
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
116180        1
116166        1
116154        1
116137        1
276723        1
Name: count, Length: 105283, dtype: int64

In [18]:
active_users = user_ratings_count[user_ratings_count > 200].index

In [19]:
ratings = ratings[ratings['user_id'].isin(active_users)]

In [20]:
ratings

,user_id,ISBN,rating
1456,277427,002542730X,10
1457,277427,0026217457,0
1458,277427,003008685X,8
1459,277427,0030615321,0
1460,277427,0060002050,0
...,...,...,...
1147612,275970,3829021860,0
1147613,275970,4770019572,0
1147614,275970,896086097,0
1147615,275970,9626340762,8


### ***Merge ratings with books***

In [21]:
ratings_with_books = ratings.merge(books, on="ISBN")

In [23]:
ratings_with_books.head(2)

,user_id,ISBN,rating,title,author,year,publisher,img_url
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...


In [24]:
ratings_with_books.shape

(487671, 8)

### ***Filter popular books (books with at least 50 ratings)***

In [26]:
book_ratings_count = ratings_with_books.groupby('title')['rating'].count().reset_index()

In [27]:
book_ratings_count.rename(columns={'rating': 'num_ratings'}, inplace=True)

In [28]:
book_ratings_count.head(2)

,title,num_ratings
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1


In [29]:
final_rating = ratings_with_books.merge(book_ratings_count, on='title')

In [30]:
final_rating.head(2)

,user_id,ISBN,rating,title,author,year,publisher,img_url,num_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,82
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...,7


In [31]:
final_rating = final_rating[final_rating['num_ratings'] >= 50]

In [32]:
final_rating.head(2)

,user_id,ISBN,rating,title,author,year,publisher,img_url,num_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,82
13,277427,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,http://images.amazon.com/images/P/0060930535.0...,133


In [33]:
final_rating.shape

(61853, 9)

In [34]:
final_rating.drop_duplicates(['user_id', 'title'], inplace=True)

In [35]:
final_rating.shape

(59850, 9)

### ***Create user-item matrix for collaborative filtering***

In [36]:
book_pivot = final_rating.pivot_table(index='title', columns='user_id', values='rating').fillna(0)

In [37]:
book_pivot.head(2)

user_id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### ***Prepare content data***

In [38]:
books_content = books.drop_duplicates('title')

In [39]:
books_content.head(2)

,ISBN,title,author,year,publisher,img_url
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...


In [40]:
books_content = books_content[books_content['title'].isin(final_rating['title'])]

### ***Combine relevant features for content analysis***

In [41]:
books_content['content_features'] = (
        books_content['title'] + ' ' + 
        books_content['author'] + ' ' + 
        books_content['publisher'].fillna('') + ' ' +
        books_content['year'].astype(str)
    )

C:\Users\emon1\AppData\Local\Temp\ipykernel_20328\3269037314.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  books_content['content_features'] = (


## ***Build recommendation models***

### ***Collaborative Filtering Model***

In [43]:
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [44]:
book_sparse = csr_matrix(book_pivot.values)

In [45]:
cf_model = NearestNeighbors(metric='cosine', algorithm='brute')

In [47]:
cf_model.fit(book_sparse)

NearestNeighbors(algorithm='brute', metric='cosine')

### ***Content-Based Model***

In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [51]:
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)

In [52]:
tfidf_matrix = tfidf.fit_transform(books_content['content_features'])

In [53]:
content_sim_matrix = cosine_similarity(tfidf_matrix)

### ***Create title to index mapping***

In [54]:
title_to_idx = pd.Series(books_content.index, index=books_content['title'])

In [55]:
title_to_idx.drop_duplicates(inplace=True)

## ***Enhanced hybrid recommendation function with image handling***

In [56]:
def hybrid_recommendations(book_title, cf_weight=0.6, cb_weight=0.4, top_n=10, fallback=True):
    """
    Generate hybrid recommendations combining collaborative and content-based filtering
    with proper image URL handling and fallback options.
    """
    try:
        recommendations = []
        
        # Collaborative Filtering Part
        cf_recs, cf_scores = [], []
        if book_title in book_pivot.index:
            book_idx = np.where(book_pivot.index == book_title)[0][0]
            distances, indices = cf_model.kneighbors(
                book_pivot.iloc[book_idx, :].values.reshape(1, -1),
                n_neighbors=top_n+1)
            
            cf_recs = book_pivot.index[indices.flatten()].tolist()
            cf_scores = (1 - distances.flatten()).tolist()
            
            if book_title in cf_recs:
                idx = cf_recs.index(book_title)
                cf_recs.pop(idx)
                cf_scores.pop(idx)
        
        # Content-Based Filtering Part
        cb_recs, cb_scores = [], []
        if book_title in title_to_idx:
            cb_idx = title_to_idx[book_title]
            sim_scores = list(enumerate(content_sim_matrix[cb_idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            sim_scores = sim_scores[1:top_n+1]  # Skip self
            
            cb_recs = [books_content['title'].iloc[i[0]] for i in sim_scores]
            cb_scores = [i[1] for i in sim_scores]
        
        # Fallback to most popular books if no recommendations
        if fallback and not cf_recs and not cb_recs:
            popular = final_rating.groupby('title')['rating'].count().sort_values(ascending=False)
            fallback_recs = popular.index.tolist()
            if book_title in fallback_recs:
                fallback_recs.remove(book_title)
            fallback_recs = fallback_recs[:top_n]
            
            for rec in fallback_recs:
                book_info = books[books['title'] == rec].iloc[0] if rec in books['title'].values else None
                if book_info is not None:
                    recommendations.append({
                        'title': rec,
                        'author': book_info['author'],
                        'year': book_info['year'],
                        'publisher': book_info['publisher'],
                        'image_url': book_info['img_url'],
                        'score': 0.7  # Default score for fallback
                    })
            return recommendations
        
        # Combine results from both methods
        combined_scores = {}
        
        # Add CF recommendations with weighted scores
        for rec, score in zip(cf_recs, cf_scores):
            combined_scores[rec] = score * cf_weight
        
        # Add CB recommendations with weighted scores
        for rec, score in zip(cb_recs, cb_scores):
            if rec in combined_scores:
                combined_scores[rec] += score * cb_weight
            else:
                combined_scores[rec] = score * cb_weight
        
        # Sort by combined score
        sorted_recs = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
        
        # Get full book details for recommendations with image URL validation
        for rec, score in sorted_recs:
            # First try to get from books_content
            book_info = books_content[books_content['title'] == rec]
            if not book_info.empty:
                book_info = book_info.iloc[0]
            else:
                # Fallback to original books data if not in books_content
                book_info = books[books['title'] == rec].iloc[0] if not books[books['title'] == rec].empty else None
                if book_info is None:
                    continue
            
            # Validate image URL
            img_url = book_info['img_url']
            if not isinstance(img_url, str) or not img_url.startswith('http'):
                img_url = "https://via.placeholder.com/150x220?text=No+Image"
            
            recommendations.append({
                'title': rec,
                'author': book_info['author'],
                'year': book_info['year'],
                'publisher': book_info['publisher'],
                'image_url': img_url,
                'score': score
            })
        
        return recommendations
    
    except Exception as e:
        print(f"Error generating recommendations: {e}")
        return []

## ***Unified recommendation function with image support***

In [57]:
def recommend_books(book_title, method='hybrid', top_n=10):
    """
    Unified recommendation function with method selection and image support
    """
    try:
        if method == 'hybrid':
            recs = hybrid_recommendations(book_title, top_n=top_n)
            print(f"\nHybrid Recommendations for '{book_title}':")
        elif method == 'collaborative':
            if book_title in book_pivot.index:
                book_idx = np.where(book_pivot.index == book_title)[0][0]
                distances, indices = cf_model.kneighbors(
                    book_pivot.iloc[book_idx, :].values.reshape(1, -1),
                    n_neighbors=top_n+1)
                
                recs = []
                for i in range(1, len(indices.flatten())):
                    title = book_pivot.index[indices.flatten()[i]]
                    # Try books_content first, then fallback to original books data
                    book_info = books_content[books_content['title'] == title]
                    if book_info.empty:
                        book_info = books[books['title'] == title]
                        if book_info.empty:
                            continue
                    
                    book_info = book_info.iloc[0]
                    
                    # Validate image URL
                    img_url = book_info['img_url']
                    if not isinstance(img_url, str) or not img_url.startswith('http'):
                        img_url = "https://via.placeholder.com/150x220?text=No+Image"
                    
                    recs.append({
                        'title': title,
                        'author': book_info['author'],
                        'year': book_info['year'],
                        'publisher': book_info['publisher'],
                        'image_url': img_url,
                        'score': (1 - distances.flatten()[i])
                    })
                print(f"\nCollaborative Filtering Recommendations for '{book_title}':")
            else:
                print(f"Book '{book_title}' not found in collaborative filtering database.")
                return []
        elif method == 'content':
            if book_title in title_to_idx:
                cb_idx = title_to_idx[book_title]
                sim_scores = list(enumerate(content_sim_matrix[cb_idx]))
                sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
                sim_scores = sim_scores[1:top_n+1]
                
                recs = []
                for i, score in sim_scores:
                    title = books_content['title'].iloc[i]
                    book_info = books_content[books_content['title'] == title].iloc[0]
                    
                    # Validate image URL
                    img_url = book_info['img_url']
                    if not isinstance(img_url, str) or not img_url.startswith('http'):
                        img_url = "https://via.placeholder.com/150x220?text=No+Image"
                    
                    recs.append({
                        'title': title,
                        'author': book_info['author'],
                        'year': book_info['year'],
                        'publisher': book_info['publisher'],
                        'image_url': img_url,
                        'score': score
                    })
                print(f"\nContent-Based Recommendations for '{book_title}':")
            else:
                print(f"Book '{book_title}' not found in content database.")
                return []
        else:
            print("Invalid method. Choose 'hybrid', 'collaborative', or 'content'.")
            return []
        
        # Print recommendations
        for i, rec in enumerate(recs[:top_n], 1):
            print(f"{i}. {rec['title']} by {rec['author']} (Score: {rec['score']:.2f})")
            print(f"   Image URL: {rec['image_url']}\n")
        
        return recs[:top_n]
    
    except Exception as e:
        print(f"Error in recommendation: {e}")
        return []

### ***Save models***

In [60]:
import pickle

In [61]:
def save_models():
    pickle.dump(cf_model, open('../models/cf_model.pkl', 'wb'))
    pickle.dump(book_pivot, open('../models/book_pivot.pkl', 'wb'))
    pickle.dump(tfidf, open('../models/tfidf_vectorizer.pkl', 'wb'))
    pickle.dump(content_sim_matrix, open('../models/content_sim_matrix.pkl', 'wb'))
    pickle.dump(title_to_idx, open('../models/title_to_idx.pkl', 'wb'))
    pickle.dump(books_content, open('../models/books_content.pkl', 'wb'))
    pickle.dump(final_rating, open('../models/final_rating.pkl', 'wb'))
    pickle.dump(books, open('../models/books_data.pkl', 'wb'))
save_models()

### ***Test the system***

In [ ]:
if __name__ == "__main__":
    # test_books = ["The Hobbit", "The Chamber", "Harry Potter and the Sorcerer's Stone", "To Kill a Mockingbird"]
    test_books = ["The Kiss"]
    
    for book in test_books:
        print("\n" + "="*80)
        print(f"Testing recommendations for: {book}")
        print("="*80)
        
        print("\nHybrid Recommendations:")
        hybrid_recs = recommend_books(book, method='hybrid')
        
        print("\nCollaborative Filtering Recommendations:")
        cf_recs = recommend_books(book, method='collaborative')
        
        print("\nContent-Based Recommendations:")
        cb_recs = recommend_books(book, method='content')


Testing recommendations for: The Kiss

Hybrid Recommendations:

Hybrid Recommendations for 'The Kiss':
1. Riptide by Catherine Coulter (Score: 0.25)
   Image URL: http://images.amazon.com/images/P/0515130966.01.LZZZZZZZ.jpg

2. Hemlock Bay by Catherine Coulter (Score: 0.24)
   Image URL: http://images.amazon.com/images/P/0515133302.01.LZZZZZZZ.jpg

3. Impulse by Catherine Coulter (Score: 0.19)
   Image URL: http://images.amazon.com/images/P/0451204301.01.LZZZZZZZ.jpg

4. The Ghost by Danielle Steel (Score: 0.17)
   Image URL: http://images.amazon.com/images/P/0440224853.01.LZZZZZZZ.jpg

5. The Reef by Nora Roberts (Score: 0.17)
   Image URL: http://images.amazon.com/images/P/051512608X.01.LZZZZZZZ.jpg

6. The First Counsel by Brad Meltzer (Score: 0.16)
   Image URL: http://images.amazon.com/images/P/0446527289.01.LZZZZZZZ.jpg

7. A Man Named Dave: A Story of Triumph and Forgiveness by David J. Pelzer (Score: 0.16)
   Image URL: http://images.amazon.com/images/P/0525945210.01.LZZZZZZZ.